In [ ]:
import pandas as pd
import os

def read_excel_sheet(filename, sheet_name):
    return pd.read_excel(filename, sheet_name=sheet_name)

# Constants
ROOT = 'D:\\IMADS\\CoffeeGrinder'
SUBDS = 'STWINonTable'
METADATA_FILE = os.path.join(ROOT, '2025 dataset planning.xlsx')
BASE_DIR = os.path.join(ROOT, SUBDS)

# Load config sheet
df = read_excel_sheet(METADATA_FILE, 'Config')

# Trim empty columns (assuming first 10 are useful)
df = df.iloc[:, :10]

# Rename and drop columns depending on the subset
COLUMN_MAP = {
    'STWINonGrinder': ('ID acq machine', 'ID acq table'),
    'STWINonTable': ('ID acq table', 'ID acq machine'),
}

rename_col, drop_col = COLUMN_MAP.get(SUBDS, (None, None))
if drop_col in df.columns:
    df = df.drop(columns=[drop_col])
if rename_col in df.columns:
    df.rename(columns={rename_col: 'ID acq'}, inplace=True)

# Drop last 3 columns (Duration, Start, End)
df = df.iloc[:, :-3]

df

In [ ]:
# ------------------------------
# Dataset Building
# ------------------------------
def build_datasets(df, BASE_DIR):
    dataset_records = []
    background_records = []
 
    df = df[df['ID acq'].notna()]
 
    for _, row in df.iterrows():
        acq_id = row['ID acq']
        file_folder = os.path.join(BASE_DIR, f'{acq_id}', '_Exported')
        mic1_file = os.path.join(file_folder, 'imp23absu_mic.wav')
        mic2_file = os.path.join(file_folder, 'imp34dt05_mic.wav')
 
        is_background = not pd.isna(row['Bkg noise']) and str(row['Bkg noise']).strip() != '\\'
 
        if is_background:
            for mic_file in [mic1_file, mic2_file]:
                background_records.append({
                    'ID acq': acq_id,
                    'file path': mic_file,
                    'pos': row['pos'],
                    'background ID': row['Bkg noise']
                })
        else:
            for mic_file in [mic1_file, mic2_file]:
                dataset_records.append({
                    'ID acq': acq_id,
                    'file path': mic_file,
                    'Model': row['Model'],
                    'Type': row['Type'],
                    'position': row['pos'],
                    'grind size': row['grinder precision (um)']
                })
 
    dataset_df = pd.DataFrame(dataset_records)
    background_df = pd.DataFrame(background_records)
 
    return dataset_df, background_df
 
dataset_df, background_df = build_datasets(df, BASE_DIR)
 
dataset_df.to_csv(os.path.join(BASE_DIR, 'dataset.csv'), index=False)
background_df.to_csv(os.path.join(BASE_DIR, 'background.csv'), index=False)

In [ ]:
background_df

In [ ]:
import librosa
import soundfile as sf
 
def export_segments(df, segments_dir, source_target_flag='source', train_test_flag='train', duration_in_seconds=5, sampling_rate =16000):
    os.makedirs(segments_dir, exist_ok=True)
 
    # Create a DataFrame to store segments, with same columns as df
    segment_columns = df.columns.tolist()
    segments_df = pd.DataFrame(columns=segment_columns)
   
    cnt = 0
    source_domain_dict = {'grind size': [0,100,200],
                          'position': ['A', 'B', 'C']}
 
    for _, row in df.iterrows():
        y, sr = librosa.load(row['file path'], sr=None)
        if sr != sampling_rate:
            print(f"Warning: {row['file path']} has a sample rate of {sr}. Expected 16000.")
        segment_length = duration_in_seconds * sr
        segments = [y[i:i + segment_length] for i in range(0, len(y), segment_length)
                    if len(y[i:i + segment_length]) == segment_length]
 
        normal_anomaly_flag = 'normal' if row['Type'] == 'Normal' else 'anomaly'
        mic_type = row['file path'].split('.')[-2].split('\\')[-1]
 
        if row['grind size'] in source_domain_dict['grind size']:
            source_target_flag = 'source'
        else:
            source_target_flag = 'target'
 
        for segment in segments:
            filename = f'section_00_{source_target_flag}_{train_test_flag}_{normal_anomaly_flag}_{str(cnt).zfill(4)}_mic_{mic_type}_pos_{row["position"]}_grind_{row["grind size"]}.wav'
            sf.write(os.path.join(segments_dir, filename), segment, sr)
            entry = {
                'ID acq': row['ID acq'],
                'file path': os.path.join('segments', filename),
                'Model': row['Model'],
                'Type': row['Type'],
                'position': row['position'],
                'grind size': row['grind size'],
                'source_target_flag': source_target_flag,
                'train_test_flag': train_test_flag,
                'normal_anomaly_flag': normal_anomaly_flag,
                'mic_type': mic_type
            }
            segments_df = pd.concat([segments_df, pd.DataFrame([entry])], ignore_index=True)
            cnt += 1
 
    return segments_df


DURATION_IN_SECONDS = 5
FS = 16000 # Hz
segments_dir = os.path.join(BASE_DIR, 'segments')

# Export segments for dataset
segments_df = export_segments(dataset_df, segments_dir,
                              source_target_flag='source', 
                              train_test_flag='train', 
                              duration_in_seconds=DURATION_IN_SECONDS, 
                              sampling_rate = FS
                              )
# save df to base folder
segments_df.to_csv(os.path.join(BASE_DIR, 'segments.csv'), index=False)
segments_df

In [ ]:
import random
import numpy as np
import librosa

WINDOW_LENGTH = DURATION_IN_SECONDS * FS
TARGET_SNR_DB = -3 # dB

# Shuffle segments randomly
segments_df = segments_df.sample(frac=1).reset_index(drop=True)
# Initialize background read heads
bkg_read_heads_dict = {'imp23absu_mic':{id: 0 for id in background_df['ID acq'].unique()},
                       'imp34dt05_mic':{id: 0 for id in background_df['ID acq'].unique()}
                    }

# Iterate over each segment
for i, row in segments_df.iterrows():
    segment_pos = row['position']
    segment_micType = row['mic_type']

    # Get list of background IDs with matching position and matching mic type
    bkg_rows_matching = background_df[(background_df['pos'] == segment_pos) & (background_df['file path'].str.contains(segment_micType))]

    legal_bkg_files = pd.DataFrame(columns=['ID acq', 'file path', 'pos', 'background ID'])
    legal_bkg_acqIDs = bkg_rows_matching['ID acq'].unique()

    # Check if there are any legal background files available for this segment
    for bck_acqID in legal_bkg_acqIDs:
        # Get background file path
        bkg_row = bkg_rows_matching[bkg_rows_matching['ID acq'] == bck_acqID]
        bck_file_path = bkg_row['file path'].values[0]

        # Load file duration (only once per ID, ideally cache this somewhere)
        try:
            bkg_duration = librosa.get_duration(path=bck_file_path) * FS
        except Exception as e:
            print(f"Failed to load {bck_file_path}: {e}")
            continue

        # Check if the read head is valid and within the file duration
        read_head = bkg_read_heads_dict[segment_micType][bck_acqID]
        if read_head != -1 and read_head + WINDOW_LENGTH <= bkg_duration:
            legal_bkg_files = pd.concat([legal_bkg_files, bkg_row], ignore_index=True)

    # Bail out if no usable background audio
    if legal_bkg_files.empty:
        print(f"No legal background files available for segment {i}.")
        continue

    # Pick random row from legal background files
    bkg_row = legal_bkg_files.sample(n=1).iloc[0]
    bck_acqID = bkg_row['ID acq']
    bck_file_path = bkg_row['file path']
    bkg_ID = bkg_row['background ID']

    # Load background audio chunk
    start_idx = bkg_read_heads_dict[segment_micType][bck_acqID]
    end_idx = start_idx + WINDOW_LENGTH

    try:
        bck_audio, _ = librosa.load(bck_file_path, sr=FS, offset=start_idx / FS, duration=DURATION_IN_SECONDS)
    except Exception as e:
        print(f"Error loading chunk from {bck_file_path}: {e}")
        continue

    # Update read head
    bkg_read_heads_dict[segment_micType][bck_acqID] = -1 if end_idx >= bkg_duration else end_idx

    # Load target audio
    try:
        file_path = os.path.join(BASE_DIR, row['file path'])
        target_audio, _ = librosa.load(file_path, sr=FS)
    except Exception as e:
        print(f"Error loading target audio from {file_path}: {e}")
        continue

    # Calculate signal powers
    signal_power = np.mean(target_audio ** 2)
    noise_power = np.mean(bck_audio ** 2)

    # Compute desired noise power for the given SNR
    target_noise_power = signal_power / (10 ** (TARGET_SNR_DB / 10))

    # Compute scaling factor for background
    scaling_factor = np.sqrt(target_noise_power / (noise_power + 1e-10))  # add epsilon to avoid division by zero
    scaled_bkg = bck_audio * scaling_factor

    # Mix
    mixed_audio = target_audio + scaled_bkg

    # add an additional column to the segments_df, which specify the background ID aplied to the segment
    segments_df.at[i, 'background ID'] = bkg_ID

    # Save mixed audio
    mixed_filename = row['file path'].replace('.wav', f'_bkg_{bkg_ID}.wav')
    mixed_filepath = os.path.join(segments_dir, os.path.basename(mixed_filename))
    sf.write(mixed_filepath, mixed_audio, FS)

# update the segments_df and save it to the base folder
segments_df.to_csv(os.path.join(BASE_DIR, 'segments.csv'), index=False)

In [ ]:
# cycle over segments_df to update anomaly file names with the substitution of 'train' with 'test'

for i, row in segments_df.iterrows():
    if 'anomaly' in row['file path']:
        old_file_name = row['file path']
        new_file_name = old_file_name.replace('train', 'test')
        segments_df.at[i, 'file path'] = new_file_name
        segments_df.at[i, 'train_test_flag'] = 'test'

        os.rename(os.path.join(BASE_DIR, old_file_name), os.path.join(BASE_DIR, new_file_name))
        #print(f"Renamed {old_file_name} to {new_file_name}")

# show some updated segments_df dataframe examples
segments_df[segments_df['train_test_flag'] == 'test'].head()

In [ ]:
# convert all column datatype to str
segments_df = segments_df.astype(str)


# concatenate values of the columns 'source_target_flag', 'grind size', 'position', and 'mic_type' into a new column 'target_label'
# using '_' as the separator

segments_df['target_label'] = segments_df['source_target_flag'] + '_' + segments_df['grind size'] + '_' + segments_df['position'] + '_' + segments_df['mic_type']
segments_df

In [ ]:
segments_df_normal = segments_df[segments_df['normal_anomaly_flag'] == 'normal']
segments_df_anomaly = segments_df[segments_df['normal_anomaly_flag'] == 'anomaly']

In [ ]:
# split into train and test sets and stratify based on the 'target_label' column
from sklearn.model_selection import train_test_split

# startify normal source
segments_df_NS = segments_df_normal[segments_df_normal['source_target_flag'] == 'source']
train_df_NS, test_df_NS = train_test_split(segments_df_NS, test_size=50, stratify=segments_df_NS['target_label'], random_state=42)

train_df_NS['target_label'].value_counts()
# test_df_NS['target_label'].value_counts() 
# test_df_NS

In [ ]:
# startify normal target
segments_df_NT = segments_df_normal[segments_df_normal['source_target_flag'] == 'target']
train_df_NT, test_df_NT = train_test_split(segments_df_NT, test_size=50, stratify=segments_df_NT['target_label'], random_state=42)

train_df_NT['target_label'].value_counts()
test_df_NT['target_label'].value_counts() 
test_df_NT

In [ ]:
# Read the CSV file into a pandas DataFrame
df = segments_df_anomaly.copy()

# Define the target number of samples
TARGET_SAMPLES = 100  # Replace with your desired number of samples

# Function to perform stratified sampling
def stratified_sample(df, target_column, target_samples):
    # Calculate the number of samples per class
    class_counts = df[target_column].value_counts()
    class_ratios = class_counts / class_counts.sum()
    samples_per_class = (class_ratios * target_samples).round().astype(int)
    
    # Sample from each class
    sampled_df = pd.concat([
        df[df[target_column] == cls].sample(n=samples_per_class[cls], random_state=42)
        for cls in class_counts.index
    ])
    
    return sampled_df

# Perform stratified sampling
test_df_AST = stratified_sample(df, 'target_label', TARGET_SAMPLES)

# Display the sampled DataFrame
test_df_AST['target_label'].value_counts()
# test_df_AST

In [ ]:
test_df = pd.concat([test_df_NS, test_df_NT, test_df_AST], ignore_index=True)
train_df = pd.concat([train_df_NS, train_df_NT], ignore_index=True)
test_df

In [ ]:
# copy all files in the train_df['file path'] to a new directory called 'train'
train_dir = os.path.join(BASE_DIR,'segments', 'train')
os.makedirs(train_dir, exist_ok=True)
for file in train_df['file path']:
    src = os.path.join(BASE_DIR, file)
    dst = os.path.join(train_dir, os.path.basename(file))
    os.system(f'copy "{src}" "{dst}"')

In [ ]:
# copy all files in the test_df['file path'] to a new directory called 'test'
import shutil  

test_dir = os.path.join(BASE_DIR, 'segments', 'test')
os.makedirs(test_dir, exist_ok=True)

# cycle over test_df and copy the files to the test directory
for i, row in test_df.iterrows():
    old_file_name = row['file path']
    new_file_name = row['file path'].replace('train', 'test')
    test_df.at[i, 'file path'] = new_file_name
    test_df.at[i, 'train_test_flag'] = 'test'

    #update the segments_df dataframe with the new file name
    segments_df.loc[segments_df['file path'] == old_file_name, 'file path'] = new_file_name
    segments_df.loc[segments_df['file path'] == old_file_name, 'train_test_flag'] = 'test'

    # rename the file in the segments directory
    old_file_path = os.path.join(BASE_DIR, old_file_name)
    new_file_path = os.path.join(BASE_DIR, new_file_name)
    os.rename(old_file_path, new_file_path)
    
    shutil.copy(new_file_path, test_dir)

In [ ]:
# count the number of element that contain the 'test' value in segments_df['file path']
test_count = segments_df['file path'].str.contains('test').sum()
print(f"Number of test files: {test_count}")